# 01 – Data Preparation and Analysis

## Objectives

* Load the raw credit card customer dataset.
* Inspect the dataset’s structure and data quality.
* Clean and transform the data for analysis.
* Save a processed dataset for use in data/processed

## Inputs

* `../data/raw/BankChurners.csv` 

## Outputs

* A cleaned Pandas DataFrame.
* Save cleaned dataset to `data/processed/cleaned_bank_churners.csv`
* Documented data-quality checks and cleaning decisions.

## Additional Comments

* Fulfills Learning Outcome 4 (LO4): Effective Data Management Practices & Data Processing Rationale.

---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [2]:
import os
current_dir = os.getcwd()
current_dir

'c:\\Projects\\credit-card-customer-churn-analysis\\jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [3]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [4]:
current_dir = os.getcwd()
current_dir

'c:\\Projects\\credit-card-customer-churn-analysis'

# Extract - Load Raw Data

Section 1 content

In [16]:
import pandas as pd
import numpy as np

# Load the raw dataset
raw_data = "data/raw/BankChurners.csv"
df = pd.read_csv(raw_data)

# Inspect shape, duplicate rows, data types, and missing values
print(f"Dataset Shape: {df.shape[0]} rows | {df.shape[1]} columns\n")

# Check for duplicate rows
print("\nDuplicate rows:")
print(df.duplicated().sum())

# Check for missing values
print("--- Missing Values ---")
print(df.isnull().sum()[df.isnull().sum() > 0])
df.head()



Dataset Shape: 10127 rows | 23 columns


Duplicate rows:
0
--- Missing Values ---
Series([], dtype: int64)


,CLIENTNUM,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,...,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2
0,768805383,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,...,12691.0,777,11914.0,1.335,1144,42,1.625,0.061,0.000093,0.99991
1,818770008,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,...,8256.0,864,7392.0,1.541,1291,33,3.714,0.105,0.000057,0.99994
2,713982108,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,...,3418.0,0,3418.0,2.594,1887,20,2.333,0.000,0.000021,0.99998
3,769911858,Existing Customer,40,F,4,High School,Unknown,Less than $40K,Blue,34,...,3313.0,2517,796.0,1.405,1171,20,2.333,0.760,0.000134,0.99987
4,709106358,Existing Customer,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,...,4716.0,0,4716.0,2.175,816,28,2.500,0.000,0.000022,0.99998


---

# Diagnose - Data Cleaning and Preparation

* Converted `Attrition_Flag` into a binary target (`churn`) assigning `1`to attributed/ churned customers and `0` to existing customers.
* Removed `CLIENTNUM` because unique identifiers contain no predictive value. Dropped the trailing `Naive_Bayes` classifier column to prevent target leakage.
* Stripped extra whitespace from all text columns to ensure accurate grouping
* Retained `"Unknown"` values across demographic variables as an explicit category to preserve the full dataset size

---

In [17]:
import os
import pandas as pd

# 1. Encode the churn target variable (1 for Attrited, 0 for Existing)
df['churn'] = df['Attrition_Flag'].apply(lambda status: 1 if status == 'Attrited Customer' else 0)

# 2. Identify and drop non-predictive features and leakage columns
leakage_cols = [col for col in df.columns if 'Naive_Bayes' in col]
drop_list = ['CLIENTNUM', 'Attrition_Flag'] + leakage_cols
df_clean = df.drop(columns=drop_list)

# 3. Clean categorical text strings
for column in df_clean.select_dtypes(include='object').columns:
    df_clean[column] = df_clean[column].str.strip()

# 4. Save processed dataset to data/processed folder
os.makedirs('data/processed', exist_ok=True)
df_clean.to_csv('data/processed/cleaned_bank_churners.csv', index=False)

# Output summary check
print(f"✅ Data processing complete! Clean dataset shape: {df_clean.shape[0]} rows | {df_clean.shape[1]} columns")
display(df_clean.head(3))

✅ Data processing complete! Clean dataset shape: 10127 rows | 20 columns


,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,churn
0,45,M,3,High School,Married,$60K - $80K,Blue,39,5,1,3,12691.0,777,11914.0,1.335,1144,42,1.625,0.061,0
1,49,F,5,Graduate,Single,Less than $40K,Blue,44,6,1,2,8256.0,864,7392.0,1.541,1291,33,3.714,0.105,0
2,51,M,3,Graduate,Married,$80K - $120K,Blue,36,4,1,0,3418.0,0,3418.0,2.594,1887,20,2.333,0.000,0


NOTE

* You may add as many sections as you want, as long as it supports your project workflow.
* All notebook's cells should be run top-down (you can't create a dynamic wherein a given point you need to go back to a previous cell to execute some task, like go back to a previous cell and refresh a variable content)

---

# Push files to Repo

##  Preparation Summary
*  Successfully ingested and audited raw customer dataset ($N = 10,127$ records, $23$ original variables).
*  Created binary target `churn` ($0 = \text{Existing Customer}, 1 = \text{Attrited Customer}$).
*  Dropped non-predictive `CLIENTNUM` identifier and raw Naive Bayes classifier columns to prevent target leakage.
*  Cleaned trailing whitespace across all categorical columns.
*  Exported processed dataset to `data/processed/cleaned_bank_churners.csv`.

## Version Control Workflow
To push all saved updates and exported dataset files to GitHub repository, execute the following commands in your VS Code terminal:

```bash
git add .
git commit -m "01 data preparation and analysis"
git push



## Next step
02_churn_visualisations_and_insights.ipynb

## Acknowledgments & Credits

### Sources & Layout Inspiration
* **Notebook Structure & Project Layout:** Inspiration for the exploratory data analysis structure and data preparation workflow was adapted from public data science repositories on Kaggle and GitHub (e.g., [Kaggle Bank Customer Churn Dataset Documentation](https://www.kaggle.com/datasets/sakshigoyal7/credit-card-customers)).
* **Assessment Framework:** Project structure aligned with the Code Institute Data Analytics Learning Outcomes (LO1–LO4).

### Tooling & Assistance
* **Python Libraries:** Built using standard open-source data science tools including `pandas`, `numpy`
* **AI & Technical Support:** Generative AI tools (Google Gemini) were utilized strictly as a technical assistant for syntax debugging, Markdown scaffolding, code optimization, and formatting advice. All core data cleaning decisions, analytical logic, and project implementations were executed independently.